In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
import numpy as np
import cv2
import os

print("TensorFlow version:", tf.__version__)

# Cell 2: Data Loading & Preparation
data_dir = "data"  # relative to notebooks folder

img_size = (160, 160)
batch_size = 16

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True
)

train_generator = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='training',
    shuffle=True
)

val_generator = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='binary',
    subset='validation',
    shuffle=False
)

print("Class indices:", train_generator.class_indices)
print("Training samples:", train_generator.samples)
print("Validation samples:", val_generator.samples)

# Cell 3: Build Model (Transfer Learning with MobileNetV2)
base_model = MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights='imagenet')
base_model.trainable = False  # Freeze base layers

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(64, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model.summary()

# Cell 4: Train the Model
epochs = 10

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=epochs
)

# Cell 5: Save model & Convert to TFLite
model.save("fire_smoke_model.h5")

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("fire_smoke_model.tflite", "wb") as f:
    f.write(tflite_model)

print("Model saved as .h5 and .tflite")

TensorFlow version: 2.21.0
Found 800 images belonging to 2 classes.
Found 199 images belonging to 2 classes.
Class indices: {'fire_images': 0, 'non_fire_images': 1}
Training samples: 800
Validation samples: 199


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 160, 160,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 80, 80,    │        864 │ input_layer_2[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 80, 80,    │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 80, 80,    │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 80, 80,    │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 80, 80,    │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 80, 80,    │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 80, 80,    │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 80, 80,    │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 80, 80,    │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 80, 80,    │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 80, 80,    │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 81, 81,    │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 40, 40,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 40, 40,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 40, 40,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 40, 40,    │      2,304 │ block_1_depthwis

 Total params: 2,340,033 (8.93 MB)

 Trainable params: 82,049 (320.50 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

Epoch 1/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 46s 791ms/step - accuracy: 0.9413 - loss: 0.1748 - val_accuracy: 0.9698 - val_loss: 0.1227
Epoch 2/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 33s 647ms/step - accuracy: 0.9750 - loss: 0.0658 - val_accuracy: 0.9648 - val_loss: 0.1163
Epoch 3/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 29s 589ms/step - accuracy: 0.9850 - loss: 0.0371 - val_accuracy: 0.9648 - val_loss: 0.1342
Epoch 4/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 30s 610ms/step - accuracy: 0.9825 - loss: 0.0619 - val_accuracy: 0.9648 - val_loss: 0.1257
Epoch 5/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 31s 619ms/step - accuracy: 0.9912 - loss: 0.0292 - val_accuracy: 0.9849 - val_loss: 0.0892
Epoch 6/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 31s 620ms/step - accuracy: 0.9950 - loss: 0.0194 - val_accuracy: 0.9799 - val_loss: 0.1018
Epoch 7/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 31s 623ms/step - accuracy: 0.9900 - loss: 0.0221 - val_accuracy: 0.9799 - val_loss: 0.1158
Epoch 8/10
50/50 ━━━━━━━━━━━━━━━━━━━━ 31s 627ms/step - accuracy: 0.9950 - loss: 0.0142 - val_accu

INFO:tensorflow:Assets written to: C:\Users\HARSH\AppData\Local\Temp\tmp4hdkj19v\assets


INFO:tensorflow:Assets written to: C:\Users\HARSH\AppData\Local\Temp\tmp4hdkj19v\assets


Saved artifact at 'C:\Users\HARSH\AppData\Local\Temp\tmp4hdkj19v'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name='keras_tensor_318')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  1872301717264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1871841938896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1871841942544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1872301718800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1871841939472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1871841941584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1871841940624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1871841940240: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1871841938704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1871841939280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train Accuracy")
plt.plot(history.history["val_accuracy"], label="Val Accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train Loss")
plt.plot(history.history["val_loss"], label="Val Loss")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.savefig("training_curves.png")
print("Graph saved")
